In [ ]:
import pandas as pd
import numpy as np
import os
import shutil
from faker import Faker
import datetime

NUM_ROWS_SALES = 1_000_000 
NUM_CUSTOMERS = 50_000     
NUM_PRODUCTS = 500          
NUM_LOCATIONS = 100        
NUM_EMPLOYEES = 50         

fake = Faker()

base_dir = "Enterprise_Big_Data_1M"
folders = {
    'fact': '1_Fact_Transactions',
    'cust': '2_Dim_Customers',
    'prod': '3_Dim_Products',
    'loc':  '4_Dim_Locations',
    'emp':  '5_Dim_Employees'
}

if os.path.exists(base_dir):
    shutil.rmtree(base_dir)

for k, v in folders.items():
    os.makedirs(os.path.join(base_dir, v), exist_ok=True)

print("🚀 Data Generation Started... (Target: 1M+ Rows, 50+ Cols)")

print("Generating Location Data...")
loc_seeds = [
    ('United States', 'North America', 'New York', 'New York', 'USD'),
    ('United States', 'North America', 'California', 'Los Angeles', 'USD'),
    ('United States', 'North America', 'Washington', 'Seattle', 'USD'),
    ('India', 'APAC', 'Maharashtra', 'Mumbai', 'INR'),
    ('India', 'APAC', 'Delhi', 'New Delhi', 'INR'),
    ('India', 'APAC', 'Karnataka', 'Bangalore', 'INR'),
    ('United Kingdom', 'Europe', 'England', 'London', 'GBP'),
    ('Germany', 'Europe', 'Berlin', 'Berlin', 'EUR'),
    ('France', 'Europe', 'Ile-de-France', 'Paris', 'EUR'),
    ('Japan', 'APAC', 'Tokyo', 'Tokyo', 'JPY'),
    ('Australia', 'APAC', 'NSW', 'Sydney', 'AUD'),
    ('Brazil', 'LATAM', 'Sao Paulo', 'Sao Paulo', 'BRL'),
    ('UAE', 'EMEA', 'Dubai', 'Dubai', 'AED')
]

loc_data = []
for i in range(NUM_LOCATIONS):
    seed = loc_seeds[i % len(loc_seeds)]
    loc_data.append({
        'Location_ID': f"LOC-{1000+i}",
        'Country': seed[0],
        'Region': seed[1],
        'State': seed[2],
        'City': seed[3],
        'Currency': seed[4],
        'Postal_Code': fake.zipcode(),
        'Timezone': fake.timezone(),
        'Warehouse_Code': f"WH-{fake.random_letter().upper()}{random.randint(1,9)}"
    })
df_loc = pd.DataFrame(loc_data)
df_loc.to_csv(f"{base_dir}/{folders['loc']}/Dim_Locations.csv", index=False)

print("Generating Product Data...")
categories = {
    'Technology': ['Phone', 'Laptop', 'Monitor', 'Headphones', 'Tablet'],
    'Furniture': ['Chair', 'Desk', 'Bookshelf', 'Sofa', 'Table'],
    'Office': ['Paper', 'Pen', 'Binder', 'Fastener', 'Label'],
    'Clothing': ['T-Shirt', 'Jeans', 'Sneakers', 'Jacket', 'Cap']
}

prod_data = []
for i in range(NUM_PRODUCTS):
    cat = np.random.choice(list(categories.keys()))
    sub = np.random.choice(categories[cat])
    prod_data.append({
        'Product_ID': f"PROD-{1000+i}",
        'Product_Name': f"{fake.word().title()} {sub}",
        'Category': cat,
        'Sub_Category': sub,
        'Brand': fake.company(),
        'Color': fake.color_name(),
        'Material': np.random.choice(['Plastic', 'Metal', 'Wood', 'Cotton', 'Leather']),
        'Base_Cost': round(np.random.uniform(10, 500), 2),
        'Supplier_Name': fake.company(),
        'Shelf_Life_Months': np.random.randint(6, 60)
    })
df_prod = pd.DataFrame(prod_data)
df_prod.to_csv(f"{base_dir}/{folders['prod']}/Dim_Products.csv", index=False)


print(f"Generating {NUM_CUSTOMERS} Customers...")
cust_ids = [f"CUST-{10000+i}" for i in range(NUM_CUSTOMERS)]
df_cust = pd.DataFrame({
    'Customer_ID': cust_ids,
    'Customer_Name': [fake.name() for _ in range(NUM_CUSTOMERS)],
    'Segment': np.random.choice(['Consumer', 'Corporate', 'Home Office'], NUM_CUSTOMERS),
    'Gender': np.random.choice(['Male', 'Female'], NUM_CUSTOMERS),
    'Age': np.random.randint(18, 75, NUM_CUSTOMERS),
    'Email_Domain': np.random.choice(['gmail.com', 'yahoo.com', 'outlook.com', 'company.com'], NUM_CUSTOMERS),
    'Loyalty_Tier': np.random.choice(['Bronze', 'Silver', 'Gold', 'Platinum'], NUM_CUSTOMERS, p=[0.5, 0.3, 0.15, 0.05]),
    'Account_Created_Date': [fake.date_between(start_date='-5y', end_date='-1y') for _ in range(NUM_CUSTOMERS)],
    'Income_Bracket': np.random.choice(['Low', 'Medium', 'High'], NUM_CUSTOMERS)
})
df_cust.to_csv(f"{base_dir}/{folders['cust']}/Dim_Customers.csv", index=False)

print("Generating Employee Data...")
df_emp = pd.DataFrame({
    'Employee_ID': [f"EMP-{100+i}" for i in range(NUM_EMPLOYEES)],
    'Sales_Rep_Name': [fake.name() for _ in range(NUM_EMPLOYEES)],
    'Department': 'Sales',
    'Joining_Date': [fake.date_between(start_date='-4y', end_date='-1y') for _ in range(NUM_EMPLOYEES)],
    'Manager': np.random.choice(['Rahul Sharma', 'Sarah Conner', 'John Wick'], NUM_EMPLOYEES),
    'Target_Quota': np.random.choice([50000, 100000, 150000], NUM_EMPLOYEES)
})
df_emp.to_csv(f"{base_dir}/{folders['emp']}/Dim_Employees.csv", index=False)

print(f"⚡ Generating 1 MILLION Rows Transaction Data (Vectorized)...")

loc_ids_col = np.random.choice(df_loc['Location_ID'], NUM_ROWS_SALES)
prod_ids_col = np.random.choice(df_prod['Product_ID'], NUM_ROWS_SALES)
cust_ids_col = np.random.choice(df_cust['Customer_ID'], NUM_ROWS_SALES)
emp_ids_col = np.random.choice(df_emp['Employee_ID'], NUM_ROWS_SALES)

start_date = np.datetime64('2023-01-01')
end_date = np.datetime64('2025-12-31')
days_range = (end_date - start_date).astype('timedelta64[D]').astype(int)
random_days = np.random.randint(0, days_range, NUM_ROWS_SALES)
order_dates = start_date + random_days.astype('timedelta64[D]')

quantities = np.random.randint(1, 20, NUM_ROWS_SALES)
unit_prices = np.random.uniform(20, 2000, NUM_ROWS_SALES).round(2) # Raw price
discount_pct = np.random.choice([0, 0.05, 0.10, 0.15, 0.20, 0.30], NUM_ROWS_SALES)
tax_pct = 0.18 

gross_sales = (quantities * unit_prices).round(2)
discount_amt = (gross_sales * discount_pct).round(2)
net_sales_before_tax = (gross_sales - discount_amt).round(2)
tax_amt = (net_sales_before_tax * tax_pct).round(2)
total_sales_value = (net_sales_before_tax + tax_amt).round(2)

cost_factor = np.random.uniform(0.5, 0.8, NUM_ROWS_SALES) 
cogs = (gross_sales * cost_factor).round(2)
profit = (net_sales_before_tax - cogs).round(2)

df_sales = pd.DataFrame({
    'Order_ID': np.arange(1000001, 1000001 + NUM_ROWS_SALES),
    'Order_Date': order_dates,
    'Ship_Date': order_dates + np.random.randint(1, 7, NUM_ROWS_SALES).astype('timedelta64[D]'),
    'Customer_ID': cust_ids_col,
    'Product_ID': prod_ids_col,
    'Location_ID': loc_ids_col,
    'Employee_ID': emp_ids_col,
    'Quantity': quantities,
    'Unit_Price': unit_prices,
    'Discount_Pct': discount_pct,
    'Discount_Amount': discount_amt,
    'Tax_Amount': tax_amt,
    'Total_Sales_Value': total_sales_value, 
    'COGS_Cost': cogs,
    'Profit': profit,
    'Payment_Mode': np.random.choice(['Credit Card', 'UPI', 'PayPal', 'Bank Transfer'], NUM_ROWS_SALES),
    'Order_Status': np.random.choice(['Delivered', 'Shipped', 'Processing', 'Cancelled'], NUM_ROWS_SALES, p=[0.7, 0.2, 0.05, 0.05]),
    'Priority': np.random.choice(['High', 'Medium', 'Low'], NUM_ROWS_SALES),
    'Return_Flag': np.random.choice([0, 1], NUM_ROWS_SALES, p=[0.92, 0.08]) # 8% Returns
})

print("Saving 1M Rows to CSV (This takes a moment)...")
df_sales.to_csv(f"{base_dir}/{folders['fact']}/Fact_Sales_1M.csv", index=False)

print("\n" + "="*50)
print("✅ SUCCESS! DATA WAREHOUSE GENERATED")
print("="*50)
print(f"📁 Root Folder: {base_dir}")
print(f"   |-- 1_Fact_Transactions (1 Million Rows) [Columns: {len(df_sales.columns)}]")
print(f"   |-- 2_Dim_Customers     (50,000 Rows)    [Columns: {len(df_cust.columns)}]")
print(f"   |-- 3_Dim_Products      (500 Rows)       [Columns: {len(df_prod.columns)}]")
print(f"   |-- 4_Dim_Locations     (100 Rows)       [Columns: {len(df_loc.columns)}]")
print(f"   |-- 5_Dim_Employees     (50 Rows)        [Columns: {len(df_emp.columns)}]")
print("\n🔥 Total Columns available for Analytics: ", 
      len(df_sales.columns) + len(df_cust.columns) + len(df_prod.columns) + len(df_loc.columns) + len(df_emp.columns))
print("Use 'ID' columns (e.g., Product_ID, Customer_ID) to JOIN tables in your Dashboard tool.")

🚀 Data Generation Started... (Target: 1M+ Rows, 50+ Cols)
Generating Location Data...
Generating Product Data...
Generating 50000 Customers...
Generating Employee Data...
⚡ Generating 1 MILLION Rows Transaction Data (Vectorized)...
Saving 1M Rows to CSV (This takes a moment)...

✅ SUCCESS! DATA WAREHOUSE GENERATED
📁 Root Folder: Enterprise_Big_Data_1M
   |-- 1_Fact_Transactions (1 Million Rows) [Columns: 19]
   |-- 2_Dim_Customers     (50,000 Rows)    [Columns: 9]
   |-- 3_Dim_Products      (500 Rows)       [Columns: 10]
   |-- 4_Dim_Locations     (100 Rows)       [Columns: 9]
   |-- 5_Dim_Employees     (50 Rows)        [Columns: 6]

🔥 Total Columns available for Analytics:  53
Use 'ID' columns (e.g., Product_ID, Customer_ID) to JOIN tables in your Dashboard tool.
